# Stock Price Prediction with LSTM
### Corrected & Optimised Version

**Bugs fixed vs. original notebook:**
1. `DataFrame.append()` (removed in pandas ≥ 2.0) → `pd.concat()`
2. Data leakage: scaler was re-fitted on test data → fit on train only, transform on test
3. `scaler = scaler.scale_` overwrote the scaler object → inverse_transform used correctly
4. yfinance MultiIndex columns → flattened immediately after download
5. Model not saved → `model.save()` call added at the end

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print('All imports OK')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
STOCK      = 'AAPL'
START      = '2010-01-01'
END        = '2024-01-01'
WINDOW     = 60       # look-back window (days)
EPOCHS     = 50
BATCH_SIZE = 32

## 1. Load Data

In [ ]:
df = yf.download(STOCK, start=START, end=END, auto_adjust=True)

# BUG FIX: yfinance ≥ 0.2 returns MultiIndex columns – flatten to single level
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(df.shape)
df.head()

In [ ]:
close = df[['Close']]

plt.figure(figsize=(14, 5))
plt.plot(close, label='Close Price')
plt.title(f'{STOCK} Closing Price History')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Train / Test Split  (before scaling — avoids data leakage)

In [ ]:
split      = int(len(close) * 0.80)
train_data = close.iloc[:split]
test_data  = close.iloc[split:]

print(f'Train rows: {len(train_data)} | Test rows: {len(test_data)}')

## 3. Scaling  (fit ONLY on train — BUG FIX: no data leakage)

In [ ]:
# BUG FIX: scaler is fit ONLY on training data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_train = scaler.fit_transform(train_data)

# For the test sequences we need the last WINDOW rows of training data
# to form the first look-back window — still using .transform(), not .fit_transform()
test_with_context = np.concatenate(
    [scaled_train[-WINDOW:], scaler.transform(test_data)],   # BUG FIX: transform only
    axis=0
)

print('scaled_train shape:', scaled_train.shape)
print('test_with_context shape:', test_with_context.shape)

## 4. Build Sequences

In [ ]:
def make_sequences(data: np.ndarray, window: int):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i - window:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = make_sequences(scaled_train,      WINDOW)
X_test,  y_test  = make_sequences(test_with_context, WINDOW)

# Reshape for LSTM: (samples, timesteps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0],   X_test.shape[1],  1)

print('X_train:', X_train.shape, '  y_train:', y_train.shape)
print('X_test: ', X_test.shape,  '  y_test: ', y_test.shape)

## 5. Build LSTM Model

In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True,  input_shape=(WINDOW, 1)),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1),
])
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

## 6. Train

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1),
]

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.10,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# Training curve
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Predictions & Inverse Scaling  (BUG FIX: scaler object preserved)

In [ ]:
y_pred = model.predict(X_test, verbose=0)

# BUG FIX: scaler object is still intact – use inverse_transform correctly
# (original code did  scaler = scaler.scale_  which destroyed the object)
y_pred_inv = scaler.inverse_transform(y_pred)
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))

plt.figure(figsize=(14, 5))
plt.plot(y_test_inv, label='Actual Price',    color='blue',  linewidth=1.2)
plt.plot(y_pred_inv, label='Predicted Price', color='red',   linewidth=1.2, linestyle='--')
plt.title(f'{STOCK} – Actual vs Predicted (Test Set)')
plt.xlabel('Days')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Next-Day Price Prediction

In [ ]:
last_window  = close.iloc[-WINDOW:].values          # last WINDOW closing prices
last_scaled  = scaler.transform(last_window)        # scale using existing scaler
X_next       = last_scaled.reshape(1, WINDOW, 1)

next_pred    = model.predict(X_next, verbose=0)
next_price   = scaler.inverse_transform(next_pred)

print(f'📈  Next-day predicted price for {STOCK}: ${next_price[0][0]:.2f}')

## 9. Save Model  (BUG FIX: model was never saved in the original)

In [ ]:
model.save('stock_dl_model.keras')
print('✅  Model saved → stock_dl_model.keras')
print('    You can now run  python app.py  to start the Flask web app.')